# torchvision resnet18 warm-up: confidence + logit magnitude check

Same isolated warm-up diagnostic as `cifarnativetrain.ipynb`, but for the
**torchvision** resnet18 (ImageNet-style 7x7 stride-2 stem + maxpool) at
224x224 input -- the architecture/config `cifar10train.ipynb`'s main pipeline
actually uses, and which matches the paper's official scratch-training
reference script (`calibration/random2/scripts/experiment_resnet.py`'s
`create_untrained_resnet18`, run via `cli/run_resnet_training.sh` with
`--lr 0.01 --weight_decay 1e-4 --epochs_noise 5`).

Unlike the CIFAR-native path, this architecture's init (`fan_out` + bias=0)
already matched its own upstream reference exactly from the start -- so the
~0.25 confidence result seen here previously was NOT explained by an init bug.
This notebook isolates just the warm-up phase (build model -> noise warm-up ->
measure confidence/logits on real CIFAR-10 test set, no downstream real-data
training) so it can be compared side by side against the now-working
CIFAR-native result.

In [1]:
%matplotlib inline
import sys, os, copy, json
import torch
import torch.nn as nn
import numpy as np
import torchvision
from torchvision.models import resnet18 as _tv_resnet18
from torch.utils.data import DataLoader

sys.path.append('..')
from src.data.degredation import get_transforms
from src.training.random_training import random_train, random_validation
from src.evaluation.inference import get_predictions_and_confidence
from src.visualization.overconfidence_plots import plot_confidence_distribution, plot_logit_distribution

import matplotlib.pyplot as plt
from custom.figure import mm, color

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams['font.family'] = 'DejaVu Sans'


def resnet18():
    """Standard torchvision resnet18 architecture, randomly initialized from scratch
    (Random2 experiment_resnet.py's create_untrained_resnet18, verbatim)."""
    model = _tv_resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 10)
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return model


In [ ]:
dir = "torchvision-224-0919"
num_net = 3

figure_dir = os.path.join("..", "figures", dir)
save_dir = os.path.join("..", "results", dir)
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

criterion = nn.CrossEntropyLoss()
num_noise, input_shape, output_size = 50000, (3, 224, 224), 10  # torchvision resnet18 input size
batch_size = 256

dataset_dir = "../dataset"
test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=False, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=True, imagenet_resize=True))
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Test: {len(test_dataset)}")


Test: 10000


c:\Users\vslab#1\Desktop\Chaewon\critical-period\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
def warmup_and_measure(tag, lr, epochs_noise=5, momentum=0.9, weight_decay=1e-4, seed_base=42):
    """torchvision resnet18, 224x224 input, noise warm-up at the given lr,
    then measure confidence + logit stats on real CIFAR-10 test set
    (no downstream real-data training -- warm-up phase only)."""
    conf_all, logit_all = [], []

    for net_idx in range(num_net):
        torch.manual_seed(seed_base + net_idx)
        np.random.seed(seed_base + net_idx)

        model = resnet18().to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

        noise_loss, noise_acc = random_validation(
            model, criterion, input_shape, num_noise, batch_size, output_size, device=device)
        print(f"[{tag}] net {net_idx} warm-up epoch 0/{epochs_noise} (baseline) "
              f"loss={noise_loss:.4f} acc={noise_acc:.4f}")

        for epoch in range(1, epochs_noise + 1):
            noise_loss, noise_acc = random_train(
                model, optimizer, criterion, input_shape, num_noise, batch_size, output_size, device=device)
            print(f"[{tag}] net {net_idx} warm-up epoch {epoch}/{epochs_noise} "
                  f"loss={noise_loss:.4f} acc={noise_acc:.4f}")

        _, conf, _, _, logit = get_predictions_and_confidence(
            model, test_loader, device=device, return_logits=True)
        per_sample_range = logit.max(axis=1) - logit.min(axis=1)

        print(f"[{tag}] net {net_idx} confidence: mean={conf.mean():.4f} std={conf.std():.4f}  "
              f"(chance=1/{output_size}={1/output_size:.4f})")
        print(f"[{tag}] net {net_idx} logit (raw): min={logit.min():.2f} max={logit.max():.2f} "
              f"mean={logit.mean():.2f} std={logit.std():.2f}")
        print(f"[{tag}] net {net_idx} logit per-sample range (max-min across classes): "
              f"mean={per_sample_range.mean():.2f} std={per_sample_range.std():.2f}")

        conf_all.append(conf)
        logit_all.append(logit)

    conf_all = np.concatenate(conf_all)
    logit_all = np.concatenate(logit_all)
    chance = 1 / output_size

    plot_confidence_distribution(conf_all, color["sky"], f"{tag} (lr={lr}, {num_net} nets)", chance_level=chance)
    plot_logit_distribution(logit_all, color["sky"], f"{tag} (lr={lr}, {num_net} nets)")

    return conf_all, logit_all


Run with `lr=0.01` -- matches `cifar10train.ipynb`'s current warm-up lr and `run_resnet_training.sh`'s official default. This is the direct comparison point for the ~0.25 confidence result seen there.

In [ ]:
conf_lr001, logit_lr001 = warmup_and_measure("torchvision_lr0.01", lr=0.01)


Optional: `lr=0.1` -- the paper Methods-text value for scratch training (differs from the CLI script's own default of 0.01). Try this too if the lr=0.01 result above doesn't converge to chance level, to see whether the same fix that worked for CIFAR-native (higher lr, now that init is correct) also helps here.

In [ ]:
conf_lr01, logit_lr01 = warmup_and_measure("torchvision_lr0.1", lr=0.1)
